# Algorithm 25

KLDM-PC-CPS with DiffCSP++ lattice projection only.

This notebook follows the lattice-only Phase 2 idea:

- keep KLDM fractional coordinates and KLDM velocity untouched
- project only the clean lattice estimate
- do the projection in DiffCSP++ `k`-space
- enforce only the lattice-family part of the oracle space group
- continue ordinary KLDM-PC sampling afterward

The default notebook is intentionally lightweight:

- `1` graph
- `1` seed
- small local tests first
- then a small faithful 800-step sampler comparison

## Source Alignment

This notebook is built against the existing repo conventions rather than a fresh CPS backend.

Relevant source paths:

- `src/kldmPlus/data/transform.py`
  - KLDM lattice preprocessing is the 6D feature representation based on logged lengths and tangent angle offsets.
- `src/kldmPlus/algorithm10_casal_chart.py`
  - `_decode_lattice_matrix(...)` and `_encode_lattice_matrix(...)` define the working lattice6 `<->` cell-matrix convention already used in KLDMPlus.
- `src/kldmPlus/symmetry/k_basis.py`
  - provides the DiffCSP++-style `cell -> k`, `k -> cell`, and family projection masks in symmetric-log-metric space.
- `src/diffcsppp/DiffCSP-PP-main/diffcsp/pl_modules/lattice/crystal_family.py`
  - upstream family constraints; the repo already mirrors the usable part in `k_basis.py`.
- `src/kldmPlus/kldm.py` and `src/kldmPlus/diffusionModels/continuous.py`
  - define the KLDM PC sampler and the lattice diffusion parameterization needed for clean-estimate CPS.

Algorithm25 notebook policy:

- use KLDMPlus lattice encode/decode helpers
- use the existing `k_basis.py` projection convention
- use CPS on the clean lattice estimate, not direct latent projection, as the main method
- use direct latent projection only as a negative-control ablation

In [ ]:
import os
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_ALLOCATOR", "platform")
os.environ.setdefault("JAX_PLATFORM_NAME", "cpu")

import math
import time
import traceback
import importlib
from dataclasses import dataclass
from pathlib import Path
from types import SimpleNamespace
from typing import Any

import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

ROOT = Path.cwd().resolve()
if not ((ROOT / 'configs').exists() and (ROOT / 'src').exists()):
    if ((ROOT.parent / 'configs').exists() and (ROOT.parent / 'src').exists()):
        ROOT = ROOT.parent

import kldmPlus.algorithm22_faithful_kldm_cps_csp as algo22_backend
import kldmPlus.algorithm25_kldm_pc_cps_lattice as algo25_backend
algo22_backend = importlib.reload(algo22_backend)
algo25_backend = importlib.reload(algo25_backend)

from kldmPlus.run_sampling_compare import SamplingCompareRunner, set_seed
from kldmPlus.sample_evaluation import build_structure_from_sample, evaluate_csp_reconstruction as evaluate_csp_reconstruction_plus
from kldm.sample_evaluation import evaluate_csp_reconstruction as evaluate_csp_reconstruction_kldm
from kldmPlus.utils.time import iter_sampling_times

Algorithm19State = algo22_backend.Algorithm19State
clean_fractional_estimate_22 = algo22_backend.clean_fractional_estimate
wrap01 = algo22_backend.wrap01
wrapdiff = algo22_backend.wrapdiff

CONFIG_PATH = ROOT / 'configs' / 'kldm_plus' / 'mp_20' / 'mp20_sampling_compare_em_vs_algorithm10_casal_chart.yaml'
with CONFIG_PATH.open('r', encoding='utf-8') as handle:
    CONFIG = yaml.safe_load(handle)

SAMPLE_SEED = int(os.environ.get('ALGO25_SEED', '2'))
GRAPH_IDS = [int(v.strip()) for v in os.environ.get('ALGO25_GRAPH_IDS', '1,2,3').split(',') if v.strip()]
PROFILE = os.environ.get('ALGO25_PROFILE', 'laptop').strip().lower()

runner = SamplingCompareRunner(config_path=CONFIG_PATH)
runner.model.eval()
set_seed(SAMPLE_SEED)

requested_num_targets = max(max(GRAPH_IDS) if GRAPH_IDS else 0, len(GRAPH_IDS), 5)
if int(runner.compare_cfg.get('num_targets', 0)) < requested_num_targets:
    runner.compare_cfg['num_targets'] = int(requested_num_targets)
    runner.compare_cfg['batch_size'] = max(int(runner.compare_cfg.get('batch_size', 0)), int(requested_num_targets))
    runner.loader, runner.lattice_transform = runner._build_loader()

full_batch = next(iter(runner.loader)).to(runner.device)
full_ptr = full_batch.ptr.tolist()
full_num_graphs = max(len(full_ptr) - 1, 0)
available_graph_ids = [int(graph_id) for graph_id in GRAPH_IDS if 1 <= int(graph_id) <= full_num_graphs]
selected_graph_indices0 = [int(graph_id) - 1 for graph_id in available_graph_ids]
selected_items = full_batch.index_select(selected_graph_indices0) if hasattr(full_batch, 'index_select') else full_batch[selected_graph_indices0]
if isinstance(selected_items, list):
    batch = full_batch.__class__.from_data_list(selected_items)
else:
    batch = selected_items
batch = batch.to(runner.device)
ptr = batch.ptr.detach().cpu().tolist()

ALGO22_CFG = algo22_backend.Algorithm22Config()
ALGO25_CFG_DEFAULT = algo25_backend.Algorithm25Config(
    total_steps=800,
    projection_start_frac=0.60,
    projection_interval=50,
    gamma_min=0.05,
    gamma_max=5.0,
    gamma_power=2.0,
    use_final_projection=False,
    preserve_lattice_volume=False,
    tau=0.25,
    n_correction_steps=1,
)
ALGO25_TOTAL_STEPS = 800
ALGO25_LOCAL_REMAINING_STEPS = [200]
ALGO25_WEIGHT_SWEEP = [0.10, 0.50, 1.00]
ALGO25_START_FRAC_SWEEP: list[int | float] = [0.60]
ALGO25_INTERVAL_SWEEP = [50]
ALGO25_GAMMA_MAX_SWEEP = [5.0]
ALGO25_SAMPLER_GRAPHS = 3
ALGO25_SAMPLER_SEEDS = [SAMPLE_SEED]
ALGO25_LOCAL_STATE_SOURCE = os.environ.get('ALGO25_LOCAL_STATE_SOURCE', 'gt_noisy_lattice').strip().lower()
ALGO25_RERUN_BASELINE = os.environ.get('ALGO25_RERUN_BASELINE', '0').strip().lower() in {'1', 'true', 'yes'}
ALGO25_DEBUG_SAMPLERS = os.environ.get('ALGO25_DEBUG_SAMPLERS', '1').strip().lower() not in {'0', 'false', 'no'}
ALGO25_DEBUG_PRINT_EVERY = int(os.environ.get('ALGO25_DEBUG_PRINT_EVERY', '25'))
ALGO25_COMPUTE_END_EVAL = os.environ.get('ALGO25_COMPUTE_END_EVAL', '0').strip().lower() in {'1', 'true', 'yes'}
ALGO25_COMPUTE_END_LATTICE = os.environ.get('ALGO25_COMPUTE_END_LATTICE', '0').strip().lower() in {'1', 'true', 'yes'}
ALGO25_COMPUTE_END_K = os.environ.get('ALGO25_COMPUTE_END_K', '0').strip().lower() in {'1', 'true', 'yes'}
ALGO25_COMPUTE_END_SYMMETRY = os.environ.get('ALGO25_COMPUTE_END_SYMMETRY', '0').strip().lower() in {'1', 'true', 'yes'}
ALGO25_CACHE: dict[tuple[Any, ...], Any] = {}

print(f'profile={PROFILE} graphs={available_graph_ids} seed={SAMPLE_SEED} total_steps={ALGO25_TOTAL_STEPS} local_state_source={ALGO25_LOCAL_STATE_SOURCE} rerun_baseline={ALGO25_RERUN_BASELINE} debug_samplers={ALGO25_DEBUG_SAMPLERS} debug_every={ALGO25_DEBUG_PRINT_EVERY} end_eval={ALGO25_COMPUTE_END_EVAL} end_lattice={ALGO25_COMPUTE_END_LATTICE} end_k={ALGO25_COMPUTE_END_K} end_symmetry={ALGO25_COMPUTE_END_SYMMETRY}')
print(f'device={runner.device} model_device={next(runner.model.parameters()).device} torch_threads={torch.get_num_threads()} interop_threads={torch.get_num_interop_threads()}')
print(f'full_batch_graphs={full_num_graphs} selected_batch_graphs={getattr(batch, "num_graphs", None)} selected_nodes={int(batch.pos.shape[0])} selected_edges={int(batch.edge_node_index.shape[1])} ptr={ptr}')


In [ ]:
@dataclass
class GraphCase:
    graph_id: int
    graph_idx0: int
    composition: dict[int, int]
    atomic_numbers: torch.Tensor
    gt_frac: torch.Tensor
    gt_l_feature: torch.Tensor
    requested_sg: int


result_tables: dict[str, pd.DataFrame] = {}


def dataframe_result(name: str, rows: list[dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    if 'PASS' not in df.columns:
        df['PASS'] = False
    if 'status' not in df.columns:
        df['status'] = np.where(df['PASS'], 'PASS', 'FAIL')
    result_tables[name] = df
    return df


def safe_display_sorted(df: pd.DataFrame, by: list[str]):
    df = df.copy()
    cols = list(df.columns)
    for key in by:
        if key not in cols:
            df[key] = np.nan
    display(df.sort_values(by).reset_index(drop=True))


def safe_metric_float(value) -> float:
    if value is None:
        return float('nan')
    if torch.is_tensor(value):
        value = value.detach().reshape(-1)
        if value.numel() == 0:
            return float('nan')
        value = value[0].item()
    try:
        return float(value)
    except Exception:
        return float('nan')


def safe_metric_bool(value) -> bool:
    if torch.is_tensor(value):
        value = value.detach().reshape(-1)
        if value.numel() == 0:
            return False
        value = value[0].item()
    return bool(value)


def finite_mean(values) -> float:
    vals = [float(v) for v in values if np.isfinite(float(v))]
    return float(np.mean(vals)) if vals else float('nan')


def finite_min(values) -> float:
    vals = [float(v) for v in values if np.isfinite(float(v))]
    return float(min(vals)) if vals else float('nan')


def graph_slice(graph_idx0: int) -> tuple[int, int]:
    return int(ptr[graph_idx0]), int(ptr[graph_idx0 + 1])


def composition_counter(values) -> dict[int, int]:
    arr = [int(v) for v in torch.as_tensor(values, dtype=torch.long).reshape(-1).tolist()]
    from collections import Counter
    return dict(sorted(Counter(arr).items()))


def graph_edge_node_index(case: GraphCase) -> torch.Tensor:
    start, end = graph_slice(case.graph_idx0)
    edge_index = batch.edge_node_index
    mask = ((edge_index[0] >= start) & (edge_index[0] < end) & (edge_index[1] >= start) & (edge_index[1] < end))
    return (edge_index[:, mask] - start).detach().clone()


def graph_tensors(graph_idx0: int) -> dict[str, Any]:
    start, end = graph_slice(graph_idx0)
    return {
        'pos': batch.pos[start:end].detach().clone(),
        'l': batch.l[graph_idx0].detach().clone().reshape(-1),
        'h': batch.atomic_numbers[start:end].detach().clone().to(torch.long),
        'sg': int(torch.as_tensor(batch.space_group).reshape(-1)[graph_idx0].item()),
    }


def load_test_graphs(graph_ids=available_graph_ids) -> list[GraphCase]:
    out = []
    for graph_idx0, graph_id in enumerate(graph_ids):
        g = graph_tensors(graph_idx0)
        out.append(GraphCase(
            graph_id=int(graph_id),
            graph_idx0=int(graph_idx0),
            composition=composition_counter(g['h']),
            atomic_numbers=g['h'],
            gt_frac=g['pos'],
            gt_l_feature=g['l'],
            requested_sg=g['sg'],
        ))
    return out


GRAPH_CASES = load_test_graphs(available_graph_ids)
print('loaded_graphs=', [g.graph_id for g in GRAPH_CASES], 'sg=', [g.requested_sg for g in GRAPH_CASES])


def make_single_graph_batch_view(case: GraphCase, *, ref_tensor: torch.Tensor | None = None):
    device = case.gt_frac.device if ref_tensor is None else ref_tensor.device
    dtype = case.gt_frac.dtype if ref_tensor is None else ref_tensor.dtype
    pos = case.gt_frac.detach().clone().to(device=device, dtype=dtype)
    l = case.gt_l_feature.detach().clone().reshape(1, -1).to(device=device, dtype=case.gt_l_feature.dtype)
    atomic_numbers = case.atomic_numbers.detach().clone().to(device=device, dtype=torch.long)
    batch_index = torch.zeros(pos.shape[0], device=device, dtype=torch.long)
    num_atoms = torch.tensor([int(pos.shape[0])], device=device, dtype=torch.long)
    edge_node_index = graph_edge_node_index(case).to(device=device)
    space_group = torch.tensor([int(case.requested_sg)], device=device, dtype=torch.long)

    class _SingleGraphBatch(SimpleNamespace):
        def to(self, device):
            out = _SingleGraphBatch()
            for key, value in self.__dict__.items():
                if torch.is_tensor(value):
                    setattr(out, key, value.to(device))
                else:
                    setattr(out, key, value)
            return out

    return _SingleGraphBatch(
        pos=pos,
        l=l,
        atomic_numbers=atomic_numbers,
        batch=batch_index,
        num_atoms=num_atoms,
        num_graphs=1,
        edge_node_index=edge_node_index,
        space_group=space_group,
    )


def build_case_structure(case: GraphCase):
    return build_structure_from_sample(case.gt_frac, case.gt_l_feature, case.atomic_numbers, lattice_transform=runner.lattice_transform)


def evaluate_arrays(case: GraphCase, pred_f: torch.Tensor, pred_l: torch.Tensor, pred_h: torch.Tensor) -> dict[str, Any]:
    result = evaluate_csp_reconstruction_plus(
        pred_f=pred_f,
        pred_l=pred_l,
        pred_a=pred_h.to(torch.long),
        target_f=case.gt_frac,
        target_l=case.gt_l_feature,
        target_a=case.atomic_numbers,
        lattice_transform=runner.lattice_transform,
        requested_space_group=case.requested_sg,
        validity_cutoff=0.5,
    )
    kldm_result = evaluate_csp_reconstruction_kldm(
        pred_f=pred_f,
        pred_l=pred_l,
        pred_a=pred_h.to(torch.long),
        target_f=case.gt_frac,
        target_l=case.gt_l_feature,
        target_a=case.atomic_numbers,
        lattice_transform=runner.lattice_transform,
    )
    standardized_frac_rmse = None
    if getattr(result, 'matcher_diagnostics', None) is not None:
        standardized_frac_rmse = getattr(result.matcher_diagnostics, 'standardized_frac_rmse', None)
    return {
        'match': bool(kldm_result.match),
        'valid': bool(kldm_result.valid),
        'rmse': kldm_result.rmse,
        'kldm_match': bool(kldm_result.match),
        'kldm_valid': bool(kldm_result.valid),
        'kldm_rmse': kldm_result.rmse,
        'plus_match': bool(result.match),
        'plus_valid': bool(result.valid),
        'plus_rmse': result.rmse,
        'frac_rmse': result.frac_rmse,
        'composition_match': result.composition_match,
        'requested_space_group_match': result.requested_space_group_match,
        'validity_reason': result.validity_reason,
        'standardized_frac_rmse': standardized_frac_rmse,
    }


def projection_family_for_sg(space_group_number: int) -> str:
    family = algo25_backend.spacegroup_to_crystal_family(int(space_group_number))
    return 'hexagonal' if family == 'trigonal' else family


def cell_lengths_angles(cell: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    cell = torch.as_tensor(cell).reshape(3, 3)
    lengths = torch.linalg.norm(cell, dim=1)
    def angle(u, v):
        denom = (torch.linalg.norm(u) * torch.linalg.norm(v)).clamp_min(1.0e-12)
        cosine = torch.clamp(torch.dot(u, v) / denom, -1.0, 1.0)
        return torch.rad2deg(torch.acos(cosine))
    alpha = angle(cell[1], cell[2])
    beta = angle(cell[0], cell[2])
    gamma = angle(cell[0], cell[1])
    return lengths, torch.stack([alpha, beta, gamma])


def cell_volume(cell: torch.Tensor) -> float:
    return float(torch.abs(torch.det(torch.as_tensor(cell).reshape(3, 3))).detach().item())


def family_constraint_metrics(cell: torch.Tensor, family: str) -> dict[str, float]:
    lengths, angles = cell_lengths_angles(cell)
    a, b, c = [float(v) for v in lengths.detach().tolist()]
    alpha, beta, gamma = [float(v) for v in angles.detach().tolist()]
    ninety = 90.0
    one_twenty = 120.0
    if family == 'triclinic':
        return {'length_equal_error': 0.0, 'angle_constraint_error': 0.0}
    if family == 'monoclinic':
        angle_error = max(abs(alpha - ninety), abs(gamma - ninety))
        return {'length_equal_error': 0.0, 'angle_constraint_error': angle_error}
    if family == 'orthorhombic':
        angle_error = max(abs(alpha - ninety), abs(beta - ninety), abs(gamma - ninety))
        return {'length_equal_error': 0.0, 'angle_constraint_error': angle_error}
    if family == 'tetragonal':
        angle_error = max(abs(alpha - ninety), abs(beta - ninety), abs(gamma - ninety))
        return {'length_equal_error': max(abs(a - b), abs(a - c) * 0.0), 'angle_constraint_error': angle_error}
    if family == 'hexagonal':
        angle_error = max(abs(alpha - ninety), abs(beta - ninety), abs(gamma - one_twenty))
        return {'length_equal_error': abs(a - b), 'angle_constraint_error': angle_error}
    if family == 'cubic':
        angle_error = max(abs(alpha - ninety), abs(beta - ninety), abs(gamma - ninety))
        return {'length_equal_error': max(abs(a - b), abs(a - c), abs(b - c)), 'angle_constraint_error': angle_error}
    return {'length_equal_error': float('nan'), 'angle_constraint_error': float('nan')}


def lattice_metric_dict(pred_l: torch.Tensor, target_l: torch.Tensor, case: GraphCase) -> dict[str, float]:
    num_atoms = int(case.atomic_numbers.shape[0])
    pred_cell = algo25_backend.lattice6_to_matrix(pred_l, num_atoms=num_atoms, lattice_transform=runner.lattice_transform)
    gt_cell = algo25_backend.lattice6_to_matrix(target_l, num_atoms=num_atoms, lattice_transform=runner.lattice_transform)
    pred_lengths, pred_angles = cell_lengths_angles(pred_cell)
    gt_lengths, gt_angles = cell_lengths_angles(gt_cell)
    pred_volume = cell_volume(pred_cell)
    gt_volume = cell_volume(gt_cell)
    return {
        'cell_matrix_rmse': float(torch.sqrt(torch.mean((pred_cell - gt_cell) ** 2)).detach().item()),
        'length_rmse': float(torch.sqrt(torch.mean((pred_lengths - gt_lengths) ** 2)).detach().item()),
        'angle_rmse_deg': float(torch.sqrt(torch.mean((pred_angles - gt_angles) ** 2)).detach().item()),
        'volume_abs_error': abs(pred_volume - gt_volume),
        'volume_rel_error': abs(pred_volume - gt_volume) / max(abs(gt_volume), 1.0e-12),
    }


def detect_spacegroup_family(pred_f: torch.Tensor, pred_l: torch.Tensor, pred_a: torch.Tensor) -> dict[str, Any]:
    try:
        structure = build_structure_from_sample(pred_f, pred_l, pred_a, lattice_transform=runner.lattice_transform)
        sga = SpacegroupAnalyzer(structure, symprec=1.0e-1, angle_tolerance=5.0)
        sg = int(sga.get_space_group_number())
        return {
            'detected_space_group': sg,
            'detected_family': algo25_backend.spacegroup_to_crystal_family(sg),
        }
    except Exception:
        return {
            'detected_space_group': None,
            'detected_family': None,
        }

In [ ]:
def _cache_key(*parts: Any) -> tuple[Any, ...]:
    return tuple(parts)


def _cached(key: tuple[Any, ...], fn):
    if key not in ALGO25_CACHE:
        ALGO25_CACHE[key] = fn()
    return ALGO25_CACHE[key]


def remaining_step_to_completed_step(remaining_step: int, total_steps: int = ALGO25_TOTAL_STEPS) -> int:
    return int(total_steps - remaining_step)


def remaining_step_to_tau(remaining_step: int, total_steps: int = ALGO25_TOTAL_STEPS) -> float:
    return float(remaining_step) / float(max(int(total_steps), 1))


ALGO25_CHECKPOINT_STEPS = tuple(sorted({remaining_step_to_completed_step(v) for v in ALGO25_LOCAL_REMAINING_STEPS} | {ALGO25_TOTAL_STEPS}))


def algo_times(case: GraphCase, t_value: float):
    dtype = case.gt_frac.dtype
    device = case.gt_frac.device
    t_graph = torch.as_tensor([[float(t_value)]], device=device, dtype=dtype)
    t_nodes = torch.full((int(case.gt_frac.shape[0]),), float(t_value), device=device, dtype=dtype)
    return t_graph, t_nodes


def make_algo_state_from_raw(*, f, v, l, atom_types, node_index, edge_node_index, t_graph, t_nodes) -> Algorithm19State:
    return Algorithm19State(
        f=f.detach().clone(),
        v=v.detach().clone(),
        l=l.detach().clone().reshape(-1),
        atom_types=atom_types.detach().clone(),
        node_index=node_index.detach().clone(),
        edge_node_index=edge_node_index.detach().clone(),
        t_graph=t_graph.detach().clone(),
        t_nodes=t_nodes.detach().clone(),
    )


def sample_gt_noisy_state(case: GraphCase, *, remaining_step: int, seed: int = SAMPLE_SEED):
    t_value = remaining_step_to_tau(remaining_step)
    batch_view = make_single_graph_batch_view(case, ref_tensor=case.gt_frac)
    set_seed(int(seed) + 1000 * int(case.graph_id) + int(remaining_step))
    t_graph, t_nodes = algo_times(case, float(t_value))
    f_t, v_t, *_ = runner.model.tdm.sample_noisy_state(t=t_nodes, f0=case.gt_frac, index=batch_view.batch)
    state = make_algo_state_from_raw(
        f=f_t,
        v=v_t,
        l=case.gt_l_feature,
        atom_types=case.atomic_numbers,
        node_index=batch_view.batch,
        edge_node_index=batch_view.edge_node_index,
        t_graph=t_graph,
        t_nodes=t_nodes,
    )
    return state


def sample_gt_noisy_lattice_state(case: GraphCase, *, remaining_step: int, seed: int = SAMPLE_SEED):
    t_value = remaining_step_to_tau(remaining_step)
    batch_view = make_single_graph_batch_view(case, ref_tensor=case.gt_frac)
    set_seed(int(seed) + 1000 * int(case.graph_id) + int(remaining_step))
    t_graph, t_nodes = algo_times(case, float(t_value))
    f_t, v_t, *_ = runner.model.tdm.sample_noisy_state(t=t_nodes, f0=case.gt_frac, index=batch_view.batch)
    t_lattice = t_graph.reshape(-1)
    set_seed(int(seed) + 2000 * int(case.graph_id) + int(remaining_step))
    l_t, _ = runner.model.diffusion_l.forward_sample(
        t=t_lattice,
        x0=case.gt_l_feature.reshape(1, -1),
        num_atoms=batch_view.num_atoms,
    )
    state = make_algo_state_from_raw(
        f=f_t,
        v=v_t,
        l=l_t.reshape(-1),
        atom_types=case.atomic_numbers,
        node_index=batch_view.batch,
        edge_node_index=batch_view.edge_node_index,
        t_graph=t_graph,
        t_nodes=t_nodes,
    )
    return state


def _clone_prepared_sampling_state(prepared: dict[str, Any]) -> dict[str, Any]:
    out = dict(prepared)
    for key, value in list(prepared.items()):
        if torch.is_tensor(value):
            out[key] = value.detach().clone()
    return out


def _pc_facit_step(prepared: dict[str, Any], times, *, tau: float = 0.25, n_correction_steps: int = 1):
    # Match facitKLDM Algorithm 4: predictor at t_now, corrector at t_next, lattice EM at t_next.
    preds_curr = runner.model.score_network(
        t=times.now.graph,
        pos=prepared['f_t'],
        v=prepared['v_t'],
        h=prepared['a_t'],
        l=prepared['l_t'],
        node_index=prepared['node_index'],
        edge_node_index=prepared['edge_node_index'],
    )
    prepared['f_t'], prepared['v_t'] = prepared['sampling_tdm'].reverse_step_predictor(
        t=times.now.nodes,
        f_t=prepared['f_t'],
        v_t=prepared['v_t'],
        pred_v=preds_curr['v'],
        index=prepared['node_index'],
        dt=times.dt,
    )

    if times.t_next_float < 1e-3:
        return

    preds_next = None
    for _ in range(max(int(n_correction_steps), 1)):
        preds_next = runner.model.score_network(
            t=times.next.graph,
            pos=prepared['f_t'],
            v=prepared['v_t'],
            h=prepared['a_t'],
            l=prepared['l_t'],
            node_index=prepared['node_index'],
            edge_node_index=prepared['edge_node_index'],
        )
        prepared['f_t'], prepared['v_t'] = prepared['sampling_tdm'].reverse_step_corrector(
            t=times.next.nodes,
            f_t=prepared['f_t'],
            v_t=prepared['v_t'],
            pred_v=preds_next['v'],
            dt=times.dt,
            index=prepared['node_index'],
            tau=float(tau),
        )

    prepared['l_t'] = prepared['sampling_diffusion_l'].reverse_step(
        t=times.next.lattice,
        x_t=prepared['l_t'],
        pred=preds_next['l'],
        dt=times.dt,
        num_atoms=prepared['batch'].num_atoms,
    )


def run_pc_trajectory_cached(case: GraphCase, *, seed: int, n_steps: int = 800, checkpoints=(400, 600, 750, 800), tau: float = 0.25, n_correction_steps: int = 1):
    def _runner():
        batch_view = make_single_graph_batch_view(case, ref_tensor=case.gt_frac)
        set_seed(int(seed) + 810000 * int(case.graph_id) + int(n_steps))
        prepared = runner.model._prepare_csp_sampling(
            batch=batch_view,
            n_steps=int(n_steps),
            t_start=1.0,
            t_final=1.0e-6,
        )
        checkpoint_map = {}
        grid = prepared['sampling_time_grid']
        times_list = list(iter_sampling_times(batch=prepared['batch'], grid=grid))
        checkpoint_set = {int(v) for v in checkpoints}
        with torch.no_grad():
            for times in times_list:
                _pc_facit_step(prepared, times, tau=float(tau), n_correction_steps=int(n_correction_steps))
                completed_step = int(times.step) + 1
                if completed_step in checkpoint_set:
                    checkpoint_map[int(completed_step)] = {
                        'prepared': _clone_prepared_sampling_state(prepared),
                        'completed_step': int(completed_step),
                    }
        final_eval = evaluate_arrays(case, prepared['f_t'], prepared['l_t'].reshape(-1), prepared['a_t'])
        return {
            'checkpoints': checkpoint_map,
            'final_f': prepared['f_t'].detach().clone(),
            'final_v': prepared['v_t'].detach().clone(),
            'final_l': prepared['l_t'].detach().clone().reshape(-1),
            'final_a': prepared['a_t'].detach().clone(),
            'final_eval': final_eval,
        }
    return _cached(_cache_key('pc_trajectory', case.graph_id, seed, n_steps, tuple(checkpoints), tau, n_correction_steps), _runner)


def extract_checkpoint_state(case: GraphCase, *, seed: int, remaining_step: int, n_steps: int = ALGO25_TOTAL_STEPS):
    completed_step = remaining_step_to_completed_step(remaining_step, total_steps=n_steps)
    cache = run_pc_trajectory_cached(case, seed=int(seed), n_steps=int(n_steps), checkpoints=ALGO25_CHECKPOINT_STEPS)
    checkpoint = cache['checkpoints'][int(completed_step)]
    prepared = _clone_prepared_sampling_state(checkpoint['prepared'])
    times_list = list(iter_sampling_times(batch=prepared['batch'], grid=prepared['sampling_time_grid']))
    idx = min(int(completed_step), max(len(times_list) - 1, 0))
    times = times_list[idx]
    state = make_algo_state_from_raw(
        f=prepared['f_t'],
        v=prepared['v_t'],
        l=prepared['l_t'],
        atom_types=prepared['a_t'],
        node_index=prepared['node_index'],
        edge_node_index=prepared['edge_node_index'],
        t_graph=times.now.graph,
        t_nodes=times.now.nodes,
    )
    return prepared, times, state


def clean_lattice_hat_from_prepared(prepared: dict[str, Any], times) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    preds = runner.model.score_network(
        t=times.now.graph,
        pos=prepared['f_t'],
        v=prepared['v_t'],
        h=prepared['a_t'],
        l=prepared['l_t'],
        node_index=prepared['node_index'],
        edge_node_index=prepared['edge_node_index'],
    )
    ell0_hat = algo25_backend.predict_clean_lattice_from_prediction(
        l_t=prepared['l_t'],
        pred_l=preds['l'],
        t_lattice=times.now.lattice,
        diffusion_l=prepared['sampling_diffusion_l'],
        num_atoms=prepared['batch'].num_atoms,
    )
    return ell0_hat.reshape(-1), preds


def clean_lattice_hat_from_state(state: Algorithm19State, *, diffusion_l: Any, t_lattice: torch.Tensor, num_atoms: torch.Tensor) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    l_in = state.l.reshape(1, -1)
    preds = runner.model.score_network(
        t=state.t_graph,
        pos=state.f,
        v=state.v,
        h=state.atom_types,
        l=l_in,
        node_index=state.node_index,
        edge_node_index=state.edge_node_index,
    )
    ell0_hat = algo25_backend.predict_clean_lattice_from_prediction(
        l_t=l_in,
        pred_l=preds['l'],
        t_lattice=t_lattice,
        diffusion_l=diffusion_l,
        num_atoms=num_atoms,
    )
    return ell0_hat.reshape(-1), preds


def get_local_bundle(case: GraphCase, *, remaining_step: int, seed: int = SAMPLE_SEED):
    def _runner():
        source = ALGO25_LOCAL_STATE_SOURCE
        if source == 'pc_checkpoint':
            prepared, times, state = extract_checkpoint_state(case, seed=int(seed), remaining_step=int(remaining_step))
            ell0_hat, preds = clean_lattice_hat_from_prepared(prepared, times)
            return {
                'source': source,
                'state': state,
                'ell0_hat': ell0_hat,
                'preds': preds,
                'diffusion_l': prepared['sampling_diffusion_l'],
                't_lattice': times.now.lattice,
                'num_atoms': prepared['batch'].num_atoms,
            }
        if source != 'gt_noisy_lattice':
            raise ValueError(f'Unsupported ALGO25_LOCAL_STATE_SOURCE={source!r}. Use "gt_noisy_lattice" or "pc_checkpoint".')
        state = sample_gt_noisy_lattice_state(case, remaining_step=int(remaining_step), seed=int(seed))
        t_lattice = state.t_graph.reshape(-1)
        num_atoms = torch.tensor([int(case.atomic_numbers.shape[0])], device=state.f.device, dtype=torch.long)
        ell0_hat, preds = clean_lattice_hat_from_state(
            state,
            diffusion_l=runner.model.diffusion_l,
            t_lattice=t_lattice,
            num_atoms=num_atoms,
        )
        return {
            'source': source,
            'state': state,
            'ell0_hat': ell0_hat,
            'preds': preds,
            'diffusion_l': runner.model.diffusion_l,
            't_lattice': t_lattice,
            'num_atoms': num_atoms,
        }
    return _cached(_cache_key('local_bundle', case.graph_id, remaining_step, seed, ALGO25_LOCAL_STATE_SOURCE), _runner)




def manual_soft_project_clean_lattice(case: GraphCase, ell0_hat: torch.Tensor, *, weight: float, preserve_volume: bool = False):
    num_atoms = int(case.atomic_numbers.shape[0])
    k_hat = algo25_backend.lattice6_to_k(ell0_hat.reshape(-1), num_atoms=num_atoms, lattice_transform=runner.lattice_transform)
    k_proj = algo25_backend.project_k_to_spacegroup_family(
        k_hat,
        space_group_number=int(case.requested_sg),
        preserve_lattice_volume=preserve_volume,
    )
    k_soft = (1.0 - float(weight)) * k_hat + float(weight) * k_proj
    if preserve_volume:
        k_soft[..., 5] = k_hat[..., 5]
    ell0_soft = algo25_backend.k_to_lattice6(k_soft, num_atoms=num_atoms, lattice_transform=runner.lattice_transform)
    return ell0_soft.reshape(-1), k_hat, k_proj, k_soft


def direct_noisy_lattice_projection(case: GraphCase, ell_t: torch.Tensor, *, preserve_volume: bool = False):
    num_atoms = int(case.atomic_numbers.shape[0])
    k_t = algo25_backend.lattice6_to_k(ell_t.reshape(-1), num_atoms=num_atoms, lattice_transform=runner.lattice_transform)
    k_proj = algo25_backend.project_k_to_spacegroup_family(
        k_t,
        space_group_number=int(case.requested_sg),
        preserve_lattice_volume=preserve_volume,
    )
    ell_proj = algo25_backend.k_to_lattice6(k_proj, num_atoms=num_atoms, lattice_transform=runner.lattice_transform)
    return ell_proj.reshape(-1), k_t, k_proj


def final_sampler_metrics(case: GraphCase, frac: torch.Tensor, lattice: torch.Tensor, atom_types: torch.Tensor) -> dict[str, Any]:
    lattice_flat = lattice.reshape(-1)
    eval_out = evaluate_arrays(case, frac, lattice_flat, atom_types)
    lattice_metrics = lattice_metric_dict(lattice_flat, case.gt_l_feature, case)
    sym = detect_spacegroup_family(frac, lattice_flat, atom_types)
    k_final = algo25_backend.lattice6_to_k(
        lattice_flat,
        num_atoms=int(case.atomic_numbers.shape[0]),
        lattice_transform=runner.lattice_transform,
    )
    final_k_violation = algo25_backend.k_family_violation(k_final, space_group_number=int(case.requested_sg))
    return {
        'eval': eval_out,
        'lattice_metrics': lattice_metrics,
        'symmetry': sym,
        'final_k_violation': final_k_violation,
    }


BASELINE_PC1000_METRIC_CACHE: dict[tuple[int, int], dict[str, Any]] = {
    # Cached directly from the user-pasted prompt table so Test 12 can iterate CPS without rerunning baseline.
    # Set ALGO25_RERUN_BASELINE=1 to regenerate these rows.
    (1, 2): {'rmse': 0.002905, 'match': True, 'valid': True, 'lattice_rmse': 0.053792, 'length_rmse': 0.068597, 'angle_rmse_deg': 0.628667, 'final_k_violation': 0.399838, 'detected_space_group': 12, 'detected_family': 'monoclinic', 'runtime_s': 64.487841},
    (1, 3): {'rmse': 0.001229, 'match': True, 'valid': True, 'lattice_rmse': 0.040293, 'length_rmse': 0.035223, 'angle_rmse_deg': 0.646725, 'final_k_violation': 0.399746, 'detected_space_group': 12, 'detected_family': 'monoclinic', 'runtime_s': 63.958167},
    (2, 2): {'rmse': 0.017590, 'match': True, 'valid': True, 'lattice_rmse': 0.076113, 'length_rmse': 0.123589, 'angle_rmse_deg': 0.407479, 'final_k_violation': 0.002919, 'detected_space_group': 4, 'detected_family': 'monoclinic', 'runtime_s': 209.335362},
    (2, 3): {'rmse': float('nan'), 'match': False, 'valid': True, 'lattice_rmse': 0.052773, 'length_rmse': 0.061662, 'angle_rmse_deg': 0.612761, 'final_k_violation': 0.004808, 'detected_space_group': 2, 'detected_family': 'triclinic', 'runtime_s': 1320.370464},
    (3, 2): {'rmse': float('nan'), 'match': False, 'valid': True, 'lattice_rmse': 2.891696, 'length_rmse': 4.047502, 'angle_rmse_deg': 23.863564, 'final_k_violation': 0.024538, 'detected_space_group': 38, 'detected_family': 'orthorhombic', 'runtime_s': 345.208223},
    (3, 3): {'rmse': float('nan'), 'match': False, 'valid': True, 'lattice_rmse': 4.304510, 'length_rmse': 6.704825, 'angle_rmse_deg': 23.873652, 'final_k_violation': 0.017236, 'detected_space_group': 38, 'detected_family': 'orthorhombic', 'runtime_s': 178.202295},
}


def cached_baseline_pc1000_metrics(case: GraphCase, seed: int) -> dict[str, Any]:
    cached = BASELINE_PC1000_METRIC_CACHE.get((int(case.graph_id), int(seed)))
    if cached is None:
        raise KeyError(
            f'No cached baseline metrics for graph={case.graph_id} seed={seed}. '
            'Set ALGO25_RERUN_BASELINE=1 to run baseline.'
        )
    return {
        'eval': {
            'rmse': cached['rmse'],
            'match': cached['match'],
            'valid': cached['valid'],
            'kldm_rmse': cached['rmse'],
            'kldm_match': cached['match'],
            'kldm_valid': cached['valid'],
            'frac_rmse': float('nan'),
            'plus_rmse': float('nan'),
            'plus_match': False,
            'plus_valid': False,
        },
        'lattice_metrics': {
            'cell_matrix_rmse': cached['lattice_rmse'],
            'length_rmse': cached['length_rmse'],
            'angle_rmse_deg': cached['angle_rmse_deg'],
        },
        'symmetry': {
            'detected_space_group': cached['detected_space_group'],
            'detected_family': cached['detected_family'],
        },
        'runtime_s': cached['runtime_s'],
        'final_k_violation': cached['final_k_violation'],
        'baseline_cache_source': 'user_prompt_table_cache',
    }


def run_algorithm25_pc_sampler(case: GraphCase, *, seed: int, config: algo25_backend.Algorithm25Config, label: str):
    key = _cache_key('algo25_pc_sampler', case.graph_id, seed, label, config)
    if ALGO25_DEBUG_SAMPLERS:
        if key in ALGO25_CACHE:
            print(f'[algo25-notebook] cache-hit sampler label={label} graph={case.graph_id} seed={seed}', flush=True)
        else:
            print(f'[algo25-notebook] start sampler label={label} graph={case.graph_id} seed={seed} total_steps={int(config.total_steps)}', flush=True)

    def _runner():
        batch_view = make_single_graph_batch_view(case, ref_tensor=case.gt_frac)
        set_seed(int(seed) + 910000 * int(case.graph_id) + abs(hash(label)) % 10000)
        t0 = time.perf_counter()
        result = algo25_backend.kldm_pc_cps_lattice_sampler(
            model=runner.model,
            batch=batch_view,
            lattice_transform=runner.lattice_transform,
            oracle_space_group=int(case.requested_sg),
            config=config,
            t_start=1.0,
            t_final=1.0e-6,
            debug_label=label if ALGO25_DEBUG_SAMPLERS else None,
            debug_print_every=ALGO25_DEBUG_PRINT_EVERY if ALGO25_DEBUG_SAMPLERS else None,
        )
        runtime_s = time.perf_counter() - t0
        metrics = final_sampler_metrics(case, result.frac_coords, result.lattice.reshape(-1), result.atom_types)
        if ALGO25_DEBUG_SAMPLERS:
            print(f'[algo25-notebook] done sampler label={label} graph={case.graph_id} seed={seed} runtime_s={runtime_s:.2f} interventions={len(result.interventions)}', flush=True)
        return {
            'result': result,
            'eval': metrics['eval'],
            'lattice_metrics': metrics['lattice_metrics'],
            'symmetry': metrics['symmetry'],
            'runtime_s': runtime_s,
            'final_k_violation': metrics['final_k_violation'],
        }
    return _cached(key, _runner)


def run_baseline_pc1000(case: GraphCase, *, seed: int, n_steps: int = ALGO25_TOTAL_STEPS):
    key = _cache_key('baseline_pc1000', case.graph_id, seed, n_steps, ALGO25_RERUN_BASELINE)
    if not ALGO25_RERUN_BASELINE and int(n_steps) == 1000:
        if ALGO25_DEBUG_SAMPLERS:
            print(f'[algo25-notebook] cached baseline graph={case.graph_id} seed={seed} n_steps={n_steps}', flush=True)
        return cached_baseline_pc1000_metrics(case, int(seed))
    if ALGO25_DEBUG_SAMPLERS:
        if key in ALGO25_CACHE:
            print(f'[algo25-notebook] cache-hit baseline graph={case.graph_id} seed={seed} n_steps={n_steps}', flush=True)
        else:
            print(f'[algo25-notebook] start baseline graph={case.graph_id} seed={seed} n_steps={n_steps}', flush=True)

    def _runner():
        batch_view = make_single_graph_batch_view(case, ref_tensor=case.gt_frac)
        set_seed(int(seed) + 920000 * int(case.graph_id) + int(n_steps))
        t0 = time.perf_counter()
        f_t, v_t, l_t, a_t = runner.model.sample_CSP_algorithm4_facit(
            n_steps=int(n_steps),
            batch=batch_view,
            t_start=1.0,
            t_final=1.0e-6,
            tau=float(ALGO25_CFG_DEFAULT.tau),
            n_correction_steps=int(ALGO25_CFG_DEFAULT.n_correction_steps),
            debug_label=f'baseline_pc_g{case.graph_id}_s{seed}' if ALGO25_DEBUG_SAMPLERS else None,
            debug_print_every=ALGO25_DEBUG_PRINT_EVERY if ALGO25_DEBUG_SAMPLERS else None,
        )
        runtime_s = time.perf_counter() - t0
        metrics = final_sampler_metrics(case, f_t, l_t.reshape(-1), a_t)
        eval_out = metrics['eval']
        lattice_metrics = metrics['lattice_metrics']
        sym = metrics['symmetry']
        final_k_violation = metrics['final_k_violation']
        if ALGO25_DEBUG_SAMPLERS:
            print(f'[algo25-notebook] done baseline graph={case.graph_id} seed={seed} runtime_s={runtime_s:.2f}', flush=True)
        return {
            'f': f_t.detach().clone(),
            'v': v_t.detach().clone(),
            'l': l_t.detach().clone().reshape(-1),
            'a': a_t.detach().clone(),
            'eval': eval_out,
            'lattice_metrics': lattice_metrics,
            'symmetry': sym,
            'runtime_s': runtime_s,
            'final_k_violation': final_k_violation,
        }
    return _cached(key, _runner)


### Test 1: KLDM Lattice Roundtrip

Verify the repo's native KLDM lattice6 `<->` cell-matrix convention is internally consistent.

This includes:
- the GT lattice
- one predicted clean lattice from a PC checkpoint
- one synthetic random valid lattice from `k`-space
- one synthetic near-degenerate but valid lattice from `k`-space

In [ ]:
rows_t1 = []
for case in GRAPH_CASES[:1]:
    device = case.gt_l_feature.device
    dtype = case.gt_l_feature.dtype
    synthetic_ks = {
        'random_k_cell': torch.tensor([0.10, -0.05, 0.08, 0.03, -0.02, 0.20], device=device, dtype=dtype),
        'near_degenerate_k_cell': torch.tensor([0.0, 0.0, 0.0, 0.0, 0.0, -0.80], device=device, dtype=dtype),
    }
    inputs = {
        'gt_lattice': case.gt_l_feature.reshape(-1),
    }
    for name, kval in synthetic_ks.items():
        cell = algo25_backend.k_to_lattice_matrix(kval)
        inputs[name] = algo25_backend.matrix_to_lattice6(cell, num_atoms=int(case.atomic_numbers.shape[0]), lattice_transform=runner.lattice_transform)

    for source, ell in inputs.items():
        cell = algo25_backend.lattice6_to_matrix(ell, num_atoms=int(case.atomic_numbers.shape[0]), lattice_transform=runner.lattice_transform)
        ell_round = algo25_backend.matrix_to_lattice6(cell, num_atoms=int(case.atomic_numbers.shape[0]), lattice_transform=runner.lattice_transform)
        metrics = lattice_metric_dict(ell_round, ell.reshape(-1), case)
        rows_t1.append({
            'test': 'algorithm25_t1_kldm_lattice_roundtrip',
            'graph': case.graph_id,
            'source': source,
            **metrics,
            'PASS': bool(metrics['length_rmse'] < 1e-5 and metrics['angle_rmse_deg'] < 1e-4),
            'status': 'INFO',
        })

safe_display_sorted(dataframe_result('algorithm25_t1_kldm_lattice_roundtrip', rows_t1), ['graph', 'source'])

### Test 2: DiffCSP++ `k` Roundtrip

Verify `lattice6 -> cell -> k -> cell -> lattice6` is numerically stable on the same small set of lattices.

In [ ]:
rows_t2 = []
for case in GRAPH_CASES[:1]:
    device = case.gt_l_feature.device
    dtype = case.gt_l_feature.dtype
    synthetic_ks = {
        'random_k_cell': torch.tensor([0.10, -0.05, 0.08, 0.03, -0.02, 0.20], device=device, dtype=dtype),
        'near_degenerate_k_cell': torch.tensor([0.0, 0.0, 0.0, 0.0, 0.0, -0.80], device=device, dtype=dtype),
    }
    inputs = {
        'gt_lattice': case.gt_l_feature.reshape(-1),
    }
    for name, kval in synthetic_ks.items():
        cell = algo25_backend.k_to_lattice_matrix(kval)
        inputs[name] = algo25_backend.matrix_to_lattice6(cell, num_atoms=int(case.atomic_numbers.shape[0]), lattice_transform=runner.lattice_transform)

    for source, ell in inputs.items():
        k = algo25_backend.lattice6_to_k(ell, num_atoms=int(case.atomic_numbers.shape[0]), lattice_transform=runner.lattice_transform)
        ell_round = algo25_backend.k_to_lattice6(k, num_atoms=int(case.atomic_numbers.shape[0]), lattice_transform=runner.lattice_transform)
        metrics = lattice_metric_dict(ell_round, ell.reshape(-1), case)
        rows_t2.append({
            'test': 'algorithm25_t2_diffcsppp_k_roundtrip',
            'graph': case.graph_id,
            'source': source,
            **metrics,
            'PASS': bool(metrics['length_rmse'] < 1e-5 and metrics['angle_rmse_deg'] < 1e-4),
            'status': 'INFO',
        })

safe_display_sorted(dataframe_result('algorithm25_t2_diffcsppp_k_roundtrip', rows_t2), ['graph', 'source'])

### Test 3: Synthetic Family Projection Correctness

Generate random `k` vectors, project them into each family, decode them, and check the expected length/angle constraints.

In [ ]:
rows_t3 = []
family_examples = [
    ('triclinic', 2),
    ('monoclinic', 15),
    ('orthorhombic', 62),
    ('tetragonal', 123),
    ('hexagonal', 194),
    ('cubic', 225),
]
rng = torch.Generator(device=GRAPH_CASES[0].gt_frac.device if GRAPH_CASES else runner.device)
rng.manual_seed(SAMPLE_SEED)
for family_name, sg in family_examples:
    k_rand = torch.randn((6,), generator=rng, device=runner.device, dtype=batch.l.dtype)
    k_proj = algo25_backend.project_k_to_spacegroup_family(k_rand, space_group_number=sg)
    cell_proj = algo25_backend.k_to_lattice_matrix(k_proj)
    metrics = family_constraint_metrics(cell_proj, projection_family_for_sg(sg))
    rows_t3.append({
        'test': 'algorithm25_t3_family_projection_synthetic',
        'family': family_name,
        'space_group': int(sg),
        'length_equal_error': metrics['length_equal_error'],
        'angle_constraint_error': metrics['angle_constraint_error'],
        'PASS': bool(metrics['length_equal_error'] < 1e-3 and metrics['angle_constraint_error'] < 1e-2),
        'status': 'INFO',
    })

safe_display_sorted(dataframe_result('algorithm25_t3_family_projection_synthetic', rows_t3), ['family'])

### Test 4: GT Lattice Identity Under Oracle Space Group

Project the GT lattice using the oracle space-group family and confirm it barely moves.

In [ ]:
rows_t4 = []
for case in GRAPH_CASES[:ALGO25_SAMPLER_GRAPHS]:
    ell_gt_pr, gt_proj_diag = algo25_backend.cps_lattice_project_clean_estimate(
        case.gt_l_feature.reshape(-1),
        num_atoms=int(case.atomic_numbers.shape[0]),
        lattice_transform=runner.lattice_transform,
        space_group_number=int(case.requested_sg),
        tau=0.0,
        gamma_min=1.0,
        gamma_max=1.0,
        gamma_power=1.0,
        projection_start_frac=0.5,
    )
    gt_id = lattice_metric_dict(ell_gt_pr, case.gt_l_feature, case)
    rows_t4.append({
        'test': 'algorithm25_t4_gt_lattice_identity',
        'graph': case.graph_id,
        'space_group': int(case.requested_sg),
        'family': projection_family_for_sg(case.requested_sg),
        **gt_id,
        'gt_projection_length_rmse': gt_id['length_rmse'],
        'gt_projection_angle_rmse': gt_id['angle_rmse_deg'],
        'gt_projection_volume_error': gt_id['volume_abs_error'],
        'gt_k_violation_before': safe_metric_float(gt_proj_diag.k_violation_before),
        'gt_k_violation_after': safe_metric_float(gt_proj_diag.k_violation_after),
        'k_violation_before': safe_metric_float(gt_proj_diag.k_violation_before),
        'k_violation_after': safe_metric_float(gt_proj_diag.k_violation_after),
        'PASS': bool(gt_id['length_rmse'] < 1e-3 and gt_id['angle_rmse_deg'] < 1e-2),
        'status': 'INFO',
    })

safe_display_sorted(dataframe_result('algorithm25_t4_gt_lattice_identity', rows_t4), ['graph'])

### Test 4B: Conventional-Cell k Identity Diagnostic

The direct GT identity test above uses the KLDM cell basis. This diagnostic asks whether the same GT structure becomes identity-preserving after spglib/Pymatgen conventional standardization.

If this passes for cubic/hexagonal cases while Test 4 fails, the DiffCSP++ k masks are probably fine and the missing piece is basis/canonical-cell handling, not the projection formula.


In [ ]:
def cell_to_cell_metric_dict(pred_cell: torch.Tensor, target_cell: torch.Tensor) -> dict[str, float]:
    pred_cell = torch.as_tensor(pred_cell).reshape(3, 3)
    target_cell = torch.as_tensor(target_cell, device=pred_cell.device, dtype=pred_cell.dtype).reshape(3, 3)
    pred_lengths, pred_angles = cell_lengths_angles(pred_cell)
    target_lengths, target_angles = cell_lengths_angles(target_cell)
    pred_metric = pred_cell @ pred_cell.transpose(-1, -2)
    target_metric = target_cell @ target_cell.transpose(-1, -2)
    pred_volume = cell_volume(pred_cell)
    target_volume = cell_volume(target_cell)
    return {
        'metric_matrix_rmse': float(torch.sqrt(torch.mean((pred_metric - target_metric) ** 2)).detach().item()),
        'length_rmse': float(torch.sqrt(torch.mean((pred_lengths - target_lengths) ** 2)).detach().item()),
        'angle_rmse_deg': float(torch.sqrt(torch.mean((pred_angles - target_angles) ** 2)).detach().item()),
        'volume_abs_error': abs(pred_volume - target_volume),
        'volume_rel_error': abs(pred_volume - target_volume) / max(abs(target_volume), 1.0e-12),
    }


rows_t4b = []
for case in GRAPH_CASES[:ALGO25_SAMPLER_GRAPHS]:
    family = projection_family_for_sg(case.requested_sg)
    gt_structure = build_case_structure(case)
    cells_to_check = []
    try:
        sga = SpacegroupAnalyzer(gt_structure, symprec=1.0e-2, angle_tolerance=5.0)
        cells_to_check.append(('spglib_refined_cell', sga.get_refined_structure()))
        cells_to_check.append(('spglib_conventional_cell', sga.get_conventional_standard_structure()))
    except Exception as exc:
        rows_t4b.append({
            'test': 'algorithm25_t4b_conventional_cell_k_identity',
            'graph': case.graph_id,
            'space_group': int(case.requested_sg),
            'family': family,
            'cell_source': 'spglib_failed',
            'error': str(exc),
            'PASS': False,
            'status': 'ERROR',
        })
        continue

    for cell_source, structure_like in cells_to_check:
        cell = torch.as_tensor(structure_like.lattice.matrix, device=case.gt_l_feature.device, dtype=case.gt_l_feature.dtype)
        k = algo25_backend.lattice_matrix_to_k(cell)
        k_proj = algo25_backend.project_k_to_spacegroup_family(
            k,
            space_group_number=int(case.requested_sg),
            preserve_lattice_volume=False,
        )
        projected_cell = algo25_backend.k_to_lattice_matrix(k_proj)
        metrics = cell_to_cell_metric_dict(projected_cell, cell)
        fam_metrics = family_constraint_metrics(projected_cell, family)
        rows_t4b.append({
            'test': 'algorithm25_t4b_conventional_cell_k_identity',
            'graph': case.graph_id,
            'space_group': int(case.requested_sg),
            'family': family,
            'cell_source': cell_source,
            **metrics,
            **fam_metrics,
            'k_violation_before': safe_metric_float(algo25_backend.k_family_violation(k, space_group_number=int(case.requested_sg))),
            'k_violation_after': safe_metric_float(algo25_backend.k_family_violation(k_proj, space_group_number=int(case.requested_sg))),
            'direct_gt_identity_passed': bool(
                len(result_tables.get('algorithm25_t4_gt_lattice_identity', pd.DataFrame()))
                and bool(result_tables['algorithm25_t4_gt_lattice_identity'].set_index('graph').get('PASS', pd.Series()).get(case.graph_id, False))
            ),
            'PASS': bool(metrics['length_rmse'] < 1e-3 and metrics['angle_rmse_deg'] < 1e-2),
            'status': 'INFO',
        })

safe_display_sorted(dataframe_result('algorithm25_t4b_conventional_cell_k_identity', rows_t4b), ['graph', 'cell_source'])


### Test 5: Wrong-Space-Group Controls

This notebook is lattice-family only, so two different controls matter:

- wrong space group from a different crystal family: should distort more
- wrong space group from the same crystal family: may look nearly identical

That second control is not a bug. It is the expected limitation of lattice-only family projection.

In [ ]:
rows_t5 = []
def choose_diff_family_sg(sg: int) -> int:
    fam = projection_family_for_sg(sg)
    for cand in [225, 194, 123, 62, 15, 2]:
        if projection_family_for_sg(cand) != fam:
            return cand
    return 225

def choose_same_family_wrong_sg(sg: int) -> int | None:
    fam = projection_family_for_sg(sg)
    for cand in range(1, 231):
        if cand != sg and projection_family_for_sg(cand) == fam:
            return cand
    return None

for case in GRAPH_CASES[:1]:
    oracle_sg = int(case.requested_sg)
    diff_family_sg = int(choose_diff_family_sg(oracle_sg))
    same_family_sg = choose_same_family_wrong_sg(oracle_sg)

    ell_oracle, _ = algo25_backend.cps_lattice_project_clean_estimate(
        case.gt_l_feature.reshape(-1),
        num_atoms=int(case.atomic_numbers.shape[0]),
        lattice_transform=runner.lattice_transform,
        space_group_number=oracle_sg,
        tau=0.0,
        gamma_min=1.0,
        gamma_max=1.0,
        gamma_power=1.0,
        projection_start_frac=0.5,
    )
    oracle_metrics = lattice_metric_dict(ell_oracle, case.gt_l_feature, case)

    ell_diff, _ = algo25_backend.cps_lattice_project_clean_estimate(
        case.gt_l_feature.reshape(-1),
        num_atoms=int(case.atomic_numbers.shape[0]),
        lattice_transform=runner.lattice_transform,
        space_group_number=diff_family_sg,
        tau=0.0,
        gamma_min=1.0,
        gamma_max=1.0,
        gamma_power=1.0,
        projection_start_frac=0.5,
    )
    diff_metrics = lattice_metric_dict(ell_diff, case.gt_l_feature, case)

    same_metrics = None
    if same_family_sg is not None:
        ell_same, _ = algo25_backend.cps_lattice_project_clean_estimate(
            case.gt_l_feature.reshape(-1),
            num_atoms=int(case.atomic_numbers.shape[0]),
            lattice_transform=runner.lattice_transform,
            space_group_number=int(same_family_sg),
            tau=0.0,
            gamma_min=1.0,
            gamma_max=1.0,
            gamma_power=1.0,
            projection_start_frac=0.5,
        )
        same_metrics = lattice_metric_dict(ell_same, case.gt_l_feature, case)

    rows_t5.append({
        'test': 'algorithm25_t5_wrong_spacegroup_controls',
        'graph': case.graph_id,
        'oracle_space_group': oracle_sg,
        'same_family_wrong_sg': int(same_family_sg) if same_family_sg is not None else None,
        'diff_family_wrong_sg': diff_family_sg,
        'oracle_length_rmse': oracle_metrics['length_rmse'],
        'oracle_angle_rmse_deg': oracle_metrics['angle_rmse_deg'],
        'same_family_length_rmse': float('nan') if same_metrics is None else same_metrics['length_rmse'],
        'same_family_angle_rmse_deg': float('nan') if same_metrics is None else same_metrics['angle_rmse_deg'],
        'diff_family_length_rmse': diff_metrics['length_rmse'],
        'diff_family_angle_rmse_deg': diff_metrics['angle_rmse_deg'],
        'PASS': bool((diff_metrics['length_rmse'] > oracle_metrics['length_rmse']) or (diff_metrics['angle_rmse_deg'] > oracle_metrics['angle_rmse_deg'])),
        'status': 'INFO',
    })

safe_display_sorted(dataframe_result('algorithm25_t5_wrong_spacegroup_controls', rows_t5), ['graph'])

### Test 6: CPS Schedule Sanity

CPS should be weak early and stronger late. This test checks the planned schedule itself before we interpret sampling results.

In [ ]:
rows_t6sched = []
remaining_steps = [600, 500, 400, 350, 300, 250, 200, 150, 100, 50]
for rem in remaining_steps:
    tau = remaining_step_to_tau(rem)
    gamma, weight = algo25_backend.cps_gamma_weight(
        tau=tau,
        projection_start_frac=float(ALGO25_CFG_DEFAULT.projection_start_frac),
        gamma_min=float(ALGO25_CFG_DEFAULT.gamma_min),
        gamma_max=float(ALGO25_CFG_DEFAULT.gamma_max),
        gamma_power=float(ALGO25_CFG_DEFAULT.gamma_power),
    )
    rows_t6sched.append({
        'test': 'algorithm25_t6_schedule_sanity',
        'remaining_step': int(rem),
        'tau': tau,
        'should_project': bool(algo25_backend.should_project_step(remaining_step=int(rem), total_steps=ALGO25_TOTAL_STEPS, projection_start_frac=float(ALGO25_CFG_DEFAULT.projection_start_frac), projection_interval=int(ALGO25_CFG_DEFAULT.projection_interval))),
        'gamma': gamma,
        'weight': weight,
        'PASS': True,
        'status': 'INFO',
    })

df_t6sched = dataframe_result('algorithm25_t6_schedule_sanity', rows_t6sched)
safe_display_sorted(df_t6sched, ['remaining_step'])

### Test 7: Clean Lattice Projection Feasibility

At selected checkpoints, project the clean lattice estimate into the oracle family and compare before/after lattice error and family violation.

In [ ]:
rows_t7 = []
for case in GRAPH_CASES[:1]:
    for remaining_step in ALGO25_LOCAL_REMAINING_STEPS:
        bundle = get_local_bundle(case, remaining_step=int(remaining_step), seed=SAMPLE_SEED)
        ell0_hat = bundle['ell0_hat']
        clean_before = lattice_metric_dict(ell0_hat, case.gt_l_feature, case)
        ell0_soft, proj_diag = algo25_backend.cps_lattice_project_clean_estimate(
            ell0_hat,
            num_atoms=int(case.atomic_numbers.shape[0]),
            lattice_transform=runner.lattice_transform,
            space_group_number=int(case.requested_sg),
            tau=remaining_step_to_tau(int(remaining_step)),
            gamma_min=float(ALGO25_CFG_DEFAULT.gamma_min),
            gamma_max=float(ALGO25_CFG_DEFAULT.gamma_max),
            gamma_power=float(ALGO25_CFG_DEFAULT.gamma_power),
            projection_start_frac=float(ALGO25_CFG_DEFAULT.projection_start_frac),
            preserve_lattice_volume=bool(ALGO25_CFG_DEFAULT.preserve_lattice_volume),
        )
        clean_after = lattice_metric_dict(ell0_soft, case.gt_l_feature, case)
        rows_t7.append({
            'test': 'algorithm25_t7_clean_lattice_projection_feasibility',
            'graph': case.graph_id,
            'state_source': bundle['source'],
            'remaining_step': int(remaining_step),
            'tau': remaining_step_to_tau(int(remaining_step)),
            'space_group': int(case.requested_sg),
            'family': projection_family_for_sg(case.requested_sg),
            'clean_lattice_rmse_before': clean_before['cell_matrix_rmse'],
            'clean_lattice_rmse_after': clean_after['cell_matrix_rmse'],
            'length_rmse_before': clean_before['length_rmse'],
            'length_rmse_after': clean_after['length_rmse'],
            'angle_rmse_before_deg': clean_before['angle_rmse_deg'],
            'angle_rmse_after_deg': clean_after['angle_rmse_deg'],
            'volume_rel_error_before': clean_before['volume_rel_error'],
            'volume_rel_error_after': clean_after['volume_rel_error'],
            'k_violation_before': safe_metric_float(proj_diag.k_violation_before),
            'k_violation_after': safe_metric_float(proj_diag.k_violation_after),
            'PASS': bool((proj_diag.k_violation_after <= proj_diag.k_violation_before + 1e-8)),
            'status': 'INFO',
        })

safe_display_sorted(dataframe_result('algorithm25_t7_clean_lattice_projection_feasibility', rows_t7), ['graph', 'remaining_step'])

### Test 8: Preserve-Volume (`k6`) Ablation

The markdown notes that shape constraints may be useful even when global scale should be preserved. This checks the cheap `preserve_lattice_volume=True` variant on the clean lattice estimate.

In [ ]:
rows_t8vol = []
for case in GRAPH_CASES[:1]:
    remaining_step = 200
    bundle = get_local_bundle(case, remaining_step=int(remaining_step), seed=SAMPLE_SEED)
    ell0_hat = bundle['ell0_hat']
    for preserve in [False, True]:
        ell0_soft, proj_diag = algo25_backend.cps_lattice_project_clean_estimate(
            ell0_hat,
            num_atoms=int(case.atomic_numbers.shape[0]),
            lattice_transform=runner.lattice_transform,
            space_group_number=int(case.requested_sg),
            tau=remaining_step_to_tau(int(remaining_step)),
            gamma_min=float(ALGO25_CFG_DEFAULT.gamma_min),
            gamma_max=float(ALGO25_CFG_DEFAULT.gamma_max),
            gamma_power=float(ALGO25_CFG_DEFAULT.gamma_power),
            projection_start_frac=float(ALGO25_CFG_DEFAULT.projection_start_frac),
            preserve_lattice_volume=bool(preserve),
        )
        metrics = lattice_metric_dict(ell0_soft, case.gt_l_feature, case)
        rows_t8vol.append({
            'test': 'algorithm25_t8_preserve_volume_ablation',
            'graph': case.graph_id,
            'state_source': bundle['source'],
            'remaining_step': int(remaining_step),
            'preserve_lattice_volume': bool(preserve),
            'clean_lattice_rmse_after': metrics['cell_matrix_rmse'],
            'length_rmse_after': metrics['length_rmse'],
            'angle_rmse_after_deg': metrics['angle_rmse_deg'],
            'volume_rel_error_after': metrics['volume_rel_error'],
            'k_violation_after': safe_metric_float(proj_diag.k_violation_after),
            'PASS': True,
            'status': 'INFO',
        })

safe_display_sorted(dataframe_result('algorithm25_t8_preserve_volume_ablation', rows_t8vol), ['graph', 'preserve_lattice_volume'])

### Test 9: Clean-vs-Noisy and `k`-Space-vs-Raw Projection Ablation

This combines two paper-faithfulness checks:

- CPS clean-estimate projection vs direct noisy-latent projection
- DiffCSP++ `k`-space projection vs a simple raw length-angle family projection

In [ ]:
rows_t9 = []
for case in GRAPH_CASES[:1]:
    remaining_step = 200
    bundle = get_local_bundle(case, remaining_step=int(remaining_step), seed=SAMPLE_SEED)
    state = bundle['state']
    ell0_hat = bundle['ell0_hat']
    ell0_soft, proj_diag = algo25_backend.cps_lattice_project_clean_estimate(
        ell0_hat,
        num_atoms=int(case.atomic_numbers.shape[0]),
        lattice_transform=runner.lattice_transform,
        space_group_number=int(case.requested_sg),
        tau=remaining_step_to_tau(int(remaining_step)),
        gamma_min=float(ALGO25_CFG_DEFAULT.gamma_min),
        gamma_max=float(ALGO25_CFG_DEFAULT.gamma_max),
        gamma_power=float(ALGO25_CFG_DEFAULT.gamma_power),
        projection_start_frac=float(ALGO25_CFG_DEFAULT.projection_start_frac),
    )
    ell_noisy_direct, _, _ = direct_noisy_lattice_projection(case, state.l.reshape(-1))
    ell_raw = algo25_backend.raw_length_angle_family_projection(
        ell0_hat,
        num_atoms=int(case.atomic_numbers.shape[0]),
        lattice_transform=runner.lattice_transform,
        space_group_number=int(case.requested_sg),
    )
    variants = {
        'clean_estimate_k_space': ell0_soft,
        'direct_noisy_lattice_projection': ell_noisy_direct,
        'raw_length_angle_projection': ell_raw,
    }
    for variant, ell in variants.items():
        metrics = lattice_metric_dict(ell, case.gt_l_feature, case)
        k_violation = algo25_backend.k_family_violation(
            algo25_backend.lattice6_to_k(ell, num_atoms=int(case.atomic_numbers.shape[0]), lattice_transform=runner.lattice_transform),
            space_group_number=int(case.requested_sg),
        )
        rows_t9.append({
            'test': 'algorithm25_t9_projection_ablation',
            'graph': case.graph_id,
            'state_source': bundle['source'],
            'remaining_step': int(remaining_step),
            'variant': variant,
            'cell_matrix_rmse': metrics['cell_matrix_rmse'],
            'length_rmse': metrics['length_rmse'],
            'angle_rmse_deg': metrics['angle_rmse_deg'],
            'volume_rel_error': metrics['volume_rel_error'],
            'k_violation_after': safe_metric_float(k_violation),
            'PASS': True,
            'status': 'INFO',
        })

safe_display_sorted(dataframe_result('algorithm25_t9_projection_ablation', rows_t9), ['graph', 'remaining_step', 'variant'])

### Test 10: One-Step Lattice-CPS State Correction

Apply the lattice-only CPS correction back to the current noisy KLDM lattice state and compare the denoiser before/after.

In [ ]:
rows_t10 = []
for case in GRAPH_CASES[:1]:
    remaining_step = 200
    bundle = get_local_bundle(case, remaining_step=int(remaining_step), seed=SAMPLE_SEED)
    state = bundle['state']
    ell0_hat = bundle['ell0_hat']
    preds_before = bundle['preds']
    f0_before = clean_fractional_estimate_22(state=state, model=runner.model, config=ALGO22_CFG)
    eval_before = evaluate_arrays(case, f0_before, ell0_hat, state.atom_types)

    ell0_soft_sched, _ = algo25_backend.cps_lattice_project_clean_estimate(
        ell0_hat,
        num_atoms=int(case.atomic_numbers.shape[0]),
        lattice_transform=runner.lattice_transform,
        space_group_number=int(case.requested_sg),
        tau=remaining_step_to_tau(int(remaining_step)),
        gamma_min=float(ALGO25_CFG_DEFAULT.gamma_min),
        gamma_max=float(ALGO25_CFG_DEFAULT.gamma_max),
        gamma_power=float(ALGO25_CFG_DEFAULT.gamma_power),
        projection_start_frac=float(ALGO25_CFG_DEFAULT.projection_start_frac),
        preserve_lattice_volume=False,
    )
    ell_t_cps, update_diag = algo25_backend.apply_lattice_cps_to_state(
        ell_t=state.l,
        ell0_hat=ell0_hat,
        ell0_soft=ell0_soft_sched,
        t_lattice=bundle['t_lattice'],
        diffusion_l=bundle['diffusion_l'],
    )
    state_after = make_algo_state_from_raw(
        f=state.f,
        v=state.v,
        l=ell_t_cps,
        atom_types=state.atom_types,
        node_index=state.node_index,
        edge_node_index=state.edge_node_index,
        t_graph=state.t_graph,
        t_nodes=state.t_nodes,
    )
    ell0_after, preds_after = clean_lattice_hat_from_state(
        state_after,
        diffusion_l=bundle['diffusion_l'],
        t_lattice=bundle['t_lattice'],
        num_atoms=bundle['num_atoms'],
    )
    f0_after = clean_fractional_estimate_22(state=state_after, model=runner.model, config=ALGO22_CFG)
    eval_after = evaluate_arrays(case, f0_after, ell0_after, state_after.atom_types)
    lattice_before = lattice_metric_dict(ell0_hat, case.gt_l_feature, case)
    lattice_after = lattice_metric_dict(ell0_after, case.gt_l_feature, case)
    rows_t10.append({
        'test': 'algorithm25_t10_one_step_state_correction',
        'graph': case.graph_id,
        'state_source': bundle['source'],
        'remaining_step': int(remaining_step),
        'tau': remaining_step_to_tau(int(remaining_step)),
        'weight_schedule': safe_metric_float(algo25_backend.cps_gamma_weight(tau=remaining_step_to_tau(int(remaining_step)), projection_start_frac=float(ALGO25_CFG_DEFAULT.projection_start_frac), gamma_min=float(ALGO25_CFG_DEFAULT.gamma_min), gamma_max=float(ALGO25_CFG_DEFAULT.gamma_max), gamma_power=float(ALGO25_CFG_DEFAULT.gamma_power))[1]),
        'lattice_state_shift_norm': safe_metric_float(update_diag.lattice_state_shift_norm),
        'clean_lattice_rmse_before': lattice_before['cell_matrix_rmse'],
        'clean_lattice_rmse_after_state_update': lattice_after['cell_matrix_rmse'],
        'predicted_f0_change': float(torch.sqrt(torch.mean(wrapdiff(f0_after, f0_before) ** 2)).detach().item()),
        'predicted_v_change': float(torch.sqrt(torch.mean((preds_after['v'] - preds_before['v']) ** 2)).detach().item()),
        'cart_rmse_before': safe_metric_float(eval_before.get('rmse')),
        'cart_rmse_after': safe_metric_float(eval_after.get('rmse')),
        'validity_before': safe_metric_bool(eval_before.get('valid')),
        'validity_after': safe_metric_bool(eval_after.get('valid')),
        'PASS': bool(safe_metric_bool(eval_after.get('valid'))),
        'status': 'INFO',
    })

safe_display_sorted(dataframe_result('algorithm25_t10_one_step_state_correction', rows_t10), ['graph', 'remaining_step'])

### Test 11: Projection Weight Sweep

A small local sweep over projection strength at one checkpoint.

In [ ]:
rows_t11 = []
for case in GRAPH_CASES[:1]:
    remaining_step = 200
    bundle = get_local_bundle(case, remaining_step=int(remaining_step), seed=SAMPLE_SEED)
    state = bundle['state']
    ell0_hat = bundle['ell0_hat']
    f0_before = clean_fractional_estimate_22(state=state, model=runner.model, config=ALGO22_CFG)
    eval_before = evaluate_arrays(case, f0_before, ell0_hat, state.atom_types)
    lattice_before = lattice_metric_dict(ell0_hat, case.gt_l_feature, case)

    for weight in ALGO25_WEIGHT_SWEEP:
        ell0_soft, _, _, _ = manual_soft_project_clean_lattice(case, ell0_hat, weight=float(weight), preserve_volume=False)
        ell_t_cps, update_diag = algo25_backend.apply_lattice_cps_to_state(
            ell_t=state.l,
            ell0_hat=ell0_hat,
            ell0_soft=ell0_soft,
            t_lattice=bundle['t_lattice'],
            diffusion_l=bundle['diffusion_l'],
        )
        state_after = make_algo_state_from_raw(
            f=state.f,
            v=state.v,
            l=ell_t_cps,
            atom_types=state.atom_types,
            node_index=state.node_index,
            edge_node_index=state.edge_node_index,
            t_graph=state.t_graph,
            t_nodes=state.t_nodes,
        )
        ell0_after, _ = clean_lattice_hat_from_state(
            state_after,
            diffusion_l=bundle['diffusion_l'],
            t_lattice=bundle['t_lattice'],
            num_atoms=bundle['num_atoms'],
        )
        f0_after = clean_fractional_estimate_22(state=state_after, model=runner.model, config=ALGO22_CFG)
        eval_after = evaluate_arrays(case, f0_after, ell0_after, state_after.atom_types)
        lattice_after = lattice_metric_dict(ell0_after, case.gt_l_feature, case)
        rows_t11.append({
            'test': 'algorithm25_t11_weight_sweep',
            'graph': case.graph_id,
            'state_source': bundle['source'],
            'remaining_step': int(remaining_step),
            'weight': float(weight),
            'state_shift_norm': safe_metric_float(update_diag.lattice_state_shift_norm),
            'clean_lattice_rmse_before': lattice_before['cell_matrix_rmse'],
            'clean_lattice_rmse_after_state_update': lattice_after['cell_matrix_rmse'],
            'cart_rmse_before': safe_metric_float(eval_before.get('rmse')),
            'cart_rmse_after': safe_metric_float(eval_after.get('rmse')),
            'validity_before': safe_metric_bool(eval_before.get('valid')),
            'validity_after': safe_metric_bool(eval_after.get('valid')),
            'PASS': True,
            'status': 'INFO',
        })

safe_display_sorted(dataframe_result('algorithm25_t11_weight_sweep', rows_t11), ['graph', 'remaining_step', 'weight'])

### Test 12: Sampler Comparison

Lightweight faithful sampler comparison:

- baseline KLDM-PC 800
- one CPS intervention at halfway
- CPS every 50 steps after halfway
- final-only projection

In [ ]:
def algo25_sweep_config(*, start_frac: float, interval: int, gamma_max: float) -> algo25_backend.Algorithm25Config:
    return algo25_backend.Algorithm25Config(
        total_steps=int(ALGO25_TOTAL_STEPS),
        projection_start_frac=float(start_frac),
        projection_interval=int(interval),
        gamma_min=float(ALGO25_CFG_DEFAULT.gamma_min),
        gamma_max=float(gamma_max),
        gamma_power=float(ALGO25_CFG_DEFAULT.gamma_power),
        use_final_projection=False,
        preserve_lattice_volume=False,
        tau=float(ALGO25_CFG_DEFAULT.tau),
        n_correction_steps=int(ALGO25_CFG_DEFAULT.n_correction_steps),
        collect_projection_diagnostics=False,
    )



def expected_projection_count(*, total_steps: int, start_frac: float, interval: int) -> int:
    if not np.isfinite(float(start_frac)) or not np.isfinite(float(interval)):
        return 0
    return sum(
        int(algo25_backend.should_project_step(
            remaining_step=int(remaining),
            total_steps=int(total_steps),
            projection_start_frac=float(start_frac),
            projection_interval=int(interval),
        ))
        for remaining in range(int(total_steps), 0, -1)
    )


def sampler_row(*, case: GraphCase, seed: int, method: str, result: dict[str, Any], start_frac, interval, gamma_max) -> dict[str, Any]:
    ev = result['eval']
    lattice_metrics = result['lattice_metrics']
    return {
        'test': 'algorithm25_t12_sampler_compare',
        'graph': case.graph_id,
        'seed': int(seed),
        'method': method,
        'start_frac': start_frac,
        'interval': interval,
        'gamma_max': gamma_max,
        'actual_space_group': int(case.requested_sg),
        'requested_space_group': int(case.requested_sg),
        'rmse': safe_metric_float(ev.get('rmse')),
        'match': safe_metric_bool(ev.get('match')),
        'valid': safe_metric_bool(ev.get('valid')),
        'frac_rmse': safe_metric_float(ev.get('frac_rmse')),
        'plus_rmse': safe_metric_float(ev.get('plus_rmse')),
        'plus_match': safe_metric_bool(ev.get('plus_match')),
        'plus_valid': safe_metric_bool(ev.get('plus_valid')),
        'lattice_rmse': lattice_metrics.get('cell_matrix_rmse', float('nan')),
        'length_rmse': lattice_metrics.get('length_rmse', float('nan')),
        'angle_rmse_deg': lattice_metrics.get('angle_rmse_deg', float('nan')),
        'final_k_violation': safe_metric_float(result['final_k_violation']),
        'detected_space_group': result['symmetry']['detected_space_group'],
        'detected_family': result['symmetry']['detected_family'],
        'num_interventions': 0 if str(method).startswith('baseline_pc') else int(len(result.get('result', SimpleNamespace(interventions=())).interventions)),
        'expected_interventions': expected_projection_count(total_steps=ALGO25_TOTAL_STEPS, start_frac=start_frac, interval=interval),
        'runtime_s': safe_metric_float(result['runtime_s']),
        'baseline_cache_source': result.get('baseline_cache_source', ''),
        'PASS': True,
        'status': 'INFO',
    }


rows_t12 = []
for case in GRAPH_CASES[:ALGO25_SAMPLER_GRAPHS]:
    for seed in ALGO25_SAMPLER_SEEDS:
        baseline = run_baseline_pc1000(case, seed=int(seed), n_steps=ALGO25_TOTAL_STEPS)
        rows_t12.append(sampler_row(
            case=case,
            seed=int(seed),
            method=f'baseline_pc{ALGO25_TOTAL_STEPS}',
            result=baseline,
            start_frac=float('nan'),
            interval=float('nan'),
            gamma_max=float('nan'),
        ))

        for start_frac in ALGO25_START_FRAC_SWEEP:
            for interval in ALGO25_INTERVAL_SWEEP:
                for gamma_max in ALGO25_GAMMA_MAX_SWEEP:
                    cfg = algo25_sweep_config(start_frac=float(start_frac), interval=int(interval), gamma_max=float(gamma_max))
                    label = f'pc{ALGO25_TOTAL_STEPS}_cps_sf{float(start_frac):.2f}_i{int(interval)}_g{float(gamma_max):.0f}'
                    out = run_algorithm25_pc_sampler(case, seed=int(seed), config=cfg, label=label)
                    rows_t12.append(sampler_row(
                        case=case,
                        seed=int(seed),
                        method=label,
                        result=out,
                        start_frac=float(start_frac),
                        interval=int(interval),
                        gamma_max=float(gamma_max),
                    ))

t12_df = dataframe_result('algorithm25_t12_sampler_compare', rows_t12)
safe_display_sorted(t12_df, ['graph', 'seed', 'method', 'start_frac', 'interval', 'gamma_max'])

summary_group_cols = ['method', 'start_frac', 'interval', 'gamma_max']
numeric_summary_cols = [
    'rmse', 'match', 'valid', 'frac_rmse', 'plus_rmse', 'plus_match', 'plus_valid',
    'lattice_rmse', 'length_rmse', 'angle_rmse_deg', 'final_k_violation', 'runtime_s', 'num_interventions', 'expected_interventions',
]
rows_t12_summary = []
for keys, group in t12_df.groupby(summary_group_cols, dropna=False):
    row = dict(zip(summary_group_cols, keys))
    row.update({
        'test': 'algorithm25_t12_sampler_summary',
        'num_runs': int(len(group)),
        'num_graphs': int(group['graph'].nunique()),
        'num_seeds': int(group['seed'].nunique()),
        'PASS': True,
        'status': 'INFO',
    })
    for col in numeric_summary_cols:
        vals = pd.to_numeric(group[col], errors='coerce') if col in group.columns else pd.Series(dtype=float)
        row[f'mean_{col}'] = float(vals.mean()) if vals.notna().any() else float('nan')
        if col in {'rmse', 'frac_rmse', 'plus_rmse', 'lattice_rmse', 'runtime_s'}:
            row[f'std_{col}'] = float(vals.std(ddof=0)) if vals.notna().any() else float('nan')
    rows_t12_summary.append(row)

safe_display_sorted(dataframe_result('algorithm25_t12_sampler_summary', rows_t12_summary), ['method', 'start_frac', 'interval', 'gamma_max'])


### Test 12B: Inspect Graph 2 Seed 3 CPS Structure

Manual inspection for the previously problematic graph 2 / seed 3 branch under the current Algorithm25 CPS settings.


In [ ]:
rows_t12b = []
inspect_graph_id = 2
inspect_seed = SAMPLE_SEED + 1
for case in GRAPH_CASES[:ALGO25_SAMPLER_GRAPHS]:
    if int(case.graph_id) != int(inspect_graph_id):
        continue
    cfg = algo25_sweep_config(
        start_frac=float(ALGO25_START_FRAC_SWEEP[0]),
        interval=int(ALGO25_INTERVAL_SWEEP[0]),
        gamma_max=float(ALGO25_GAMMA_MAX_SWEEP[0]),
    )
    label = f'pc{ALGO25_TOTAL_STEPS}_cps_sf{float(cfg.projection_start_frac):.2f}_i{int(cfg.projection_interval)}_g{float(cfg.gamma_max):.0f}'
    out = run_algorithm25_pc_sampler(case, seed=int(inspect_seed), config=cfg, label=label)
    result = out['result']
    final_structure = build_structure_from_sample(result.frac_coords, result.lattice.reshape(-1), result.atom_types, lattice_transform=runner.lattice_transform)
    gt_structure = build_case_structure(case)
    final_cell = algo25_backend.lattice6_to_matrix(result.lattice.reshape(-1), num_atoms=int(case.atomic_numbers.shape[0]), lattice_transform=runner.lattice_transform)
    gt_cell = algo25_backend.lattice6_to_matrix(case.gt_l_feature.reshape(-1), num_atoms=int(case.atomic_numbers.shape[0]), lattice_transform=runner.lattice_transform)
    final_cart = torch.as_tensor(final_structure.cart_coords, dtype=case.gt_frac.dtype)
    gt_cart = torch.as_tensor(gt_structure.cart_coords, dtype=case.gt_frac.dtype)
    cart_pair_dists = torch.cdist(final_cart, gt_cart)
    nearest_cart = cart_pair_dists.min(dim=1).values
    rows_t12b.append({
        'test': 'algorithm25_t12b_bad_case_inspection',
        'graph': int(case.graph_id),
        'seed': int(inspect_seed),
        'method': label,
        'actual_space_group': int(case.requested_sg),
        'detected_space_group': out['symmetry']['detected_space_group'],
        'detected_family': out['symmetry']['detected_family'],
        'match': safe_metric_bool(out['eval'].get('match')),
        'valid': safe_metric_bool(out['eval'].get('valid')),
        'rmse': safe_metric_float(out['eval'].get('rmse')),
        'lattice_rmse': safe_metric_float(out['lattice_metrics'].get('cell_matrix_rmse')),
        'length_rmse': safe_metric_float(out['lattice_metrics'].get('length_rmse')),
        'angle_rmse_deg': safe_metric_float(out['lattice_metrics'].get('angle_rmse_deg')),
        'final_volume': float(final_structure.volume),
        'gt_volume': float(gt_structure.volume),
        'volume_ratio': float(final_structure.volume / max(gt_structure.volume, 1.0e-12)),
        'final_density': float(final_structure.density),
        'gt_density': float(gt_structure.density),
        'density_ratio': float(final_structure.density / max(gt_structure.density, 1.0e-12)),
        'nearest_cart_mean': float(nearest_cart.mean().item()),
        'nearest_cart_max': float(nearest_cart.max().item()),
        'final_lattice': np.round(final_cell.detach().cpu().numpy(), 5).tolist(),
        'gt_lattice': np.round(gt_cell.detach().cpu().numpy(), 5).tolist(),
        'final_frac_coords': np.round(result.frac_coords.detach().cpu().numpy(), 5).tolist(),
        'PASS': True,
        'status': 'INFO',
    })

safe_display_sorted(dataframe_result('algorithm25_t12b_bad_case_inspection', rows_t12b), ['graph', 'seed', 'method'])


### Test 13: Verdict

High-level summary combining the small local checks and the lightweight sampler comparison.

# Debug

Late projection debug. Here `completed_start_frac=0.70` means we wait until 70% of the reverse trajectory is complete, i.e. project only when `remaining_tau <= 0.30`.

This is intentionally small by default: it picks one direct-GT-identity-safe graph if available, one seed, and compares cached baseline against late lattice-CPS starts.


In [ ]:
# Debug Test D1: late lattice-CPS start sweep
# Laptop default: one safe graph, one seed, no baseline rerun.
ALGO25_DEBUG_LATE_COMPLETED_START_FRACS = [0.70, 0.80, 0.90]
ALGO25_DEBUG_LATE_INTERVAL = 50
ALGO25_DEBUG_LATE_GAMMA_MAX = 5.0
ALGO25_DEBUG_LATE_GRAPH_LIMIT = int(os.environ.get('ALGO25_DEBUG_LATE_GRAPH_LIMIT', '1'))
ALGO25_DEBUG_LATE_SEED_LIMIT = int(os.environ.get('ALGO25_DEBUG_LATE_SEED_LIMIT', '1'))


def _debug_late_cases() -> list[GraphCase]:
    t4 = result_tables.get('algorithm25_t4_gt_lattice_identity', pd.DataFrame())
    if len(t4) and 'PASS' in t4.columns:
        safe_ids = set(int(v) for v in t4.loc[t4['PASS'] == True, 'graph'].tolist())
        safe_cases = [case for case in GRAPH_CASES[:ALGO25_SAMPLER_GRAPHS] if int(case.graph_id) in safe_ids]
        if safe_cases:
            return safe_cases[:ALGO25_DEBUG_LATE_GRAPH_LIMIT]
    # Fallback: prefer graph 2 if present because it is the current direct-basis sanity pass case.
    graph2 = [case for case in GRAPH_CASES[:ALGO25_SAMPLER_GRAPHS] if int(case.graph_id) == 2]
    if graph2:
        return graph2[:ALGO25_DEBUG_LATE_GRAPH_LIMIT]
    return GRAPH_CASES[:ALGO25_DEBUG_LATE_GRAPH_LIMIT]


rows_debug_late = []
for case in _debug_late_cases():
    for seed in ALGO25_SAMPLER_SEEDS[:ALGO25_DEBUG_LATE_SEED_LIMIT]:
        baseline = run_baseline_pc1000(case, seed=int(seed), n_steps=ALGO25_TOTAL_STEPS)
        rows_debug_late.append(sampler_row(
            case=case,
            seed=int(seed),
            method=f'debug_cached_baseline_pc{ALGO25_TOTAL_STEPS}',
            result=baseline,
            start_frac=float('nan'),
            interval=float('nan'),
            gamma_max=float('nan'),
        ) | {
            'test': 'algorithm25_debug_d1_late_projection_sweep',
            'completed_start_frac': float('nan'),
            'remaining_start_tau': float('nan'),
        })

        for completed_start_frac in ALGO25_DEBUG_LATE_COMPLETED_START_FRACS:
            remaining_start_tau = 1.0 - float(completed_start_frac)
            cfg = algo25_sweep_config(
                start_frac=float(remaining_start_tau),
                interval=int(ALGO25_DEBUG_LATE_INTERVAL),
                gamma_max=float(ALGO25_DEBUG_LATE_GAMMA_MAX),
            )
            label = f'debug_late_pc{ALGO25_TOTAL_STEPS}_completed{float(completed_start_frac):.2f}_i{int(ALGO25_DEBUG_LATE_INTERVAL)}_g{float(ALGO25_DEBUG_LATE_GAMMA_MAX):.0f}'
            out = run_algorithm25_pc_sampler(case, seed=int(seed), config=cfg, label=label)
            rows_debug_late.append(sampler_row(
                case=case,
                seed=int(seed),
                method=label,
                result=out,
                start_frac=float(remaining_start_tau),
                interval=int(ALGO25_DEBUG_LATE_INTERVAL),
                gamma_max=float(ALGO25_DEBUG_LATE_GAMMA_MAX),
            ) | {
                'test': 'algorithm25_debug_d1_late_projection_sweep',
                'completed_start_frac': float(completed_start_frac),
                'remaining_start_tau': float(remaining_start_tau),
            })

debug_late_df = dataframe_result('algorithm25_debug_d1_late_projection_sweep', rows_debug_late)
safe_display_sorted(debug_late_df, ['graph', 'seed', 'completed_start_frac', 'method'])

summary_cols = ['method', 'completed_start_frac', 'remaining_start_tau', 'start_frac', 'interval', 'gamma_max']
metric_cols = ['rmse', 'match', 'valid', 'frac_rmse', 'lattice_rmse', 'length_rmse', 'angle_rmse_deg', 'final_k_violation', 'num_interventions', 'expected_interventions', 'runtime_s']
rows_debug_late_summary = []
for keys, group in debug_late_df.groupby(summary_cols, dropna=False):
    row = dict(zip(summary_cols, keys))
    row.update({
        'test': 'algorithm25_debug_d1_late_projection_summary',
        'num_runs': int(len(group)),
        'num_graphs': int(group['graph'].nunique()),
        'num_seeds': int(group['seed'].nunique()),
        'PASS': True,
        'status': 'INFO',
    })
    for col in metric_cols:
        vals = pd.to_numeric(group[col], errors='coerce') if col in group.columns else pd.Series(dtype=float)
        row[f'mean_{col}'] = float(vals.mean()) if vals.notna().any() else float('nan')
    rows_debug_late_summary.append(row)

safe_display_sorted(dataframe_result('algorithm25_debug_d1_late_projection_summary', rows_debug_late_summary), ['completed_start_frac', 'method'])


In [ ]:
rows_t13 = []
for case in GRAPH_CASES[:1]:
    t1 = result_tables.get('algorithm25_t1_kldm_lattice_roundtrip', pd.DataFrame())
    t2 = result_tables.get('algorithm25_t2_diffcsppp_k_roundtrip', pd.DataFrame())
    t4 = result_tables.get('algorithm25_t4_gt_lattice_identity', pd.DataFrame())
    t7 = result_tables.get('algorithm25_t7_clean_lattice_projection_feasibility', pd.DataFrame())
    t10 = result_tables.get('algorithm25_t10_one_step_state_correction', pd.DataFrame())
    t12 = result_tables.get('algorithm25_t12_sampler_compare', pd.DataFrame())

    foundation_ok = bool(len(t1) and len(t2) and len(t4) and t1['PASS'].all() and t2['PASS'].all() and t4['PASS'].all())
    local_k_improves = bool(len(t7) and np.any(t7['k_violation_after'] <= t7['k_violation_before'] + 1e-8))
    local_lattice_help = bool(len(t7) and np.any(t7['clean_lattice_rmse_after'] <= t7['clean_lattice_rmse_before'] + 1e-8))
    state_stable = bool(len(t10) and np.any(t10['validity_after'] == True))

    baseline = t12[t12['method'] == f'baseline_pc{ALGO25_TOTAL_STEPS}'] if len(t12) else pd.DataFrame()
    other = t12[t12['method'] != f'baseline_pc{ALGO25_TOTAL_STEPS}'] if len(t12) else pd.DataFrame()
    baseline_lattice = finite_mean(baseline['lattice_rmse']) if len(baseline) else float('nan')
    best_cps_lattice = finite_min(other['lattice_rmse']) if len(other) else float('nan')
    sampler_beats_baseline = bool(np.isfinite(best_cps_lattice) and np.isfinite(baseline_lattice) and best_cps_lattice < baseline_lattice)

    if foundation_ok and local_k_improves and local_lattice_help and sampler_beats_baseline:
        recommendation = 'promising_algorithm25_lattice_cps_sampler'
    elif foundation_ok and local_k_improves and local_lattice_help:
        recommendation = 'local_lattice_positive_not_sampler_positive_yet'
    elif foundation_ok and local_k_improves:
        recommendation = 'projection_enforces_family_but_sampler_gain_unclear'
    else:
        recommendation = 'stop_and_audit_lattice_mapping_or_schedule'

    rows_t13.append({
        'test': 'algorithm25_t13_verdict',
        'graph': case.graph_id,
        'foundation_ok': foundation_ok,
        'local_k_violation_improves': local_k_improves,
        'local_lattice_rmse_help': local_lattice_help,
        'state_correction_stable': state_stable,
        'sampler_beats_baseline_on_lattice_rmse': sampler_beats_baseline,
        'recommendation': recommendation,
        'PASS': foundation_ok,
        'status': 'INFO',
    })

safe_display_sorted(dataframe_result('algorithm25_t13_verdict', rows_t13), ['graph'])

## Suggested Run Order

1. setup cells
2. `### Test 1` through `### Test 6`
3. `### Test 7` through `### Test 11`
4. `### Test 12`
5. `### Test 13`

Laptop-ready defaults:

- `### Test 7` to `### Test 11` use `ALGO25_LOCAL_STATE_SOURCE=gt_noisy_lattice` by default
- `### Test 12` runs only `baseline_pc800` plus the Algorithm25 PC-CPS schedule sweep
- set `ALGO25_LOCAL_STATE_SOURCE=pc_checkpoint` only if you want the fully faithful but much slower local diagnostics

If the sampler comparison is still slow, run `### Test 12` last or skip it on the first pass.